In [1]:
import pandas as pd
import re
import string
from itertools import combinations
from collections import Counter

import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

print("Imports complete.")


Imports complete.


In [2]:
# Load Data

df = pd.read_csv("comments_cleaned.csv")

keep_cols = [c for c in [
    "video_id", "video_title", "author",
    "comment_published_at", "comment_like_count", "clean_comment"
] if c in df.columns]

df = df[keep_cols].copy()
df = df.dropna(subset=["clean_comment"]).reset_index(drop=True)

print(f"Loaded {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df.head(3)


Loaded 6,241 rows
Columns: ['video_id', 'video_title', 'author', 'comment_published_at', 'comment_like_count', 'clean_comment']


,video_id,video_title,author,comment_published_at,comment_like_count,clean_comment
0,laZpTO7IFtA,Is 67 just brain rot?,@ExoticNibbles2,2025-10-14T22:24:09Z,7902,nothing makes me feel older then watching a 15...
1,laZpTO7IFtA,Is 67 just brain rot?,@andrewbunnell7576,2025-10-13T20:12:09Z,7692,6 x 7 = 42 the answer to life the universe and...
2,laZpTO7IFtA,Is 67 just brain rot?,@ColinPaddock,2025-10-14T06:39:54Z,6194,all i know is 6 is afraid of 7.


In [3]:
# Create network_text, Lowercase, Remove Punctuation and Extra Spaces

df["network_text"] = df["clean_comment"].astype(str)

# Lowercase (clean_comment already is, but enforce)
df["network_text"] = df["network_text"].str.lower()

# Remove punctuation except hyphens and underscores (needed for phrase joining later)
df["network_text"] = df["network_text"].apply(
    lambda t: re.sub(r"[^\w\s\-]", " ", t)
)

# Collapse extra whitespace
df["network_text"] = df["network_text"].apply(
    lambda t: re.sub(r"\s+", " ", t).strip()
)

print("Preprocessing step 3 complete.")
print("Sample:", df["network_text"].iloc[0])


Preprocessing step 3 complete.
Sample: nothing makes me feel older then watching a 15 minute video breaking down slang the young kids use these days


In [4]:
# Standardise Terms and Convert Multi-word Phrases

# Order matters: more specific patterns first.
# Each tuple is (regex_pattern, replacement).
TEXT_REPLACEMENTS = [
    # --- Standardise variant spellings ---
    (r"\bbrain[\s\-]?rot\b",              "brainrot"),
    (r"\bdoom[\s\-]?scrolling\b",         "doomscrolling"),
    (r"\btik[\s\-]?tok\b",                "tiktok"),
    (r"\bno[\s\-]?cap\b",                 "nocap"),
    (r"\bover[\s\-]?stimulation\b",       "overstimulation"),
    (r"\bshort[\s\-]?form\b",             "shortform"),
    (r"\blong[\s\-]?form\b",              "longform"),

    # --- Convert multi-word phrases to single tokens ---
    (r"\bgen(?:eration)?\s+alpha\b",      "genalpha"),   # before gen z
    (r"\bgen(?:eration)?\s+z(?:ee)?\b",   "genz"),
    (r"\bgen(?:eration)?\s+x\b",          "genx"),
    (r"\bgen(?:eration)?\s+a\b",          "genalpha"),
    (r"\bipad\s+kids?\b",                 "ipadkids"),
    (r"\battention\s+span\b",             "attention_span"),
    (r"\bsocial\s+media\b",               "social_media"),
    (r"\bscreen\s+time\b",                "screen_time"),
    (r"\bshort\s+video\b",                "short_video"),
    (r"\bbrain\s+damage\b",               "brain_damage"),
    (r"\bbrain\s+worm\b",                 "brainworm"),
    (r"\bmental\s+health\b",              "mental_health"),
]

def apply_replacements(text: str) -> str:
    for pattern, replacement in TEXT_REPLACEMENTS:
        text = re.sub(pattern, replacement, text)
    return text

df["network_text"] = df["network_text"].apply(apply_replacements)

# Remove hyphens that remain (they were protected during punctuation removal)
df["network_text"] = df["network_text"].apply(
    lambda t: re.sub(r"-", " ", t)
)
df["network_text"] = df["network_text"].apply(
    lambda t: re.sub(r"\s+", " ", t).strip()
)

print("Standardisation complete.")
print("Sample:", df["network_text"].iloc[3])


Standardisation complete.
Sample: having completely no context for it before this video i thought it was a solo version of 69


In [5]:
# Slang and theme dictionary

# Keys are the exact tokens to detect in network_text.
# Values are the theme/category label for the node.
TERM_THEMES = {
    # Brainrot / overstimulation
    "brainrot":          "brainrot",
    "doomscrolling":     "brainrot",
    "overstimulation":   "brainrot",
    "mindless":          "brainrot",
    "zombie":            "brainrot",
    "rotting":           "brainrot",
    "brainworm":         "brainrot",

    # Platform / addiction
    "tiktok":            "platform",
    "reels":             "platform",
    "shorts":            "platform",
    "instagram":         "platform",
    "scrolling":         "platform",
    "binge":             "platform",
    "algorithm":         "platform",
    "social_media":      "platform",
    "screen_time":       "platform",

    # Meme slang
    "skibidi":           "meme_slang",
    "rizz":              "meme_slang",
    "gyatt":             "meme_slang",
    "sigma":             "meme_slang",
    "sus":               "meme_slang",
    "slay":              "meme_slang",
    "sheesh":            "meme_slang",
    "yeet":              "meme_slang",
    "nocap":             "meme_slang",
    "based":             "meme_slang",
    "ratio":             "meme_slang",
    "vibe":              "meme_slang",
    "goat":              "meme_slang",

    # Generational identity
    "genz":              "generational",
    "genalpha":          "generational",
    "genx":              "generational",
    "millennial":        "generational",
    "boomer":            "generational",
    "zoomer":            "generational",
    "ipadkids":          "generational",

    # Cognitive / behavioural
    "adhd":              "cognitive",
    "dopamine":          "cognitive",
    "attention_span":    "cognitive",
    "focus":             "cognitive",
    "addiction":         "cognitive",
    "distraction":       "cognitive",
    "anxiety":           "cognitive",
    "rewiring":          "cognitive",
    "boredom":           "cognitive",
    "mental_health":     "cognitive",

    # Language / culture
    "slang":             "language",
    "meme":              "language",
    "viral":             "language",
    "trend":             "language",
    "culture":           "language",
    "language":          "language",
    "shortform":         "content",
    "longform":          "content",
}

VOCABULARY = set(TERM_THEMES.keys())
print(f"Vocabulary: {len(VOCABULARY)} terms across {len(set(TERM_THEMES.values()))} themes")
print(f"Themes: {sorted(set(TERM_THEMES.values()))}")


Vocabulary: 54 terms across 7 themes
Themes: ['brainrot', 'cognitive', 'content', 'generational', 'language', 'meme_slang', 'platform']


In [6]:
# Detect relevant terms per comment

def detect_terms(text: str) -> list:
    # Split on whitespace to get exact tokens; no partial matches
    tokens = set(text.split())
    return sorted(tokens & VOCABULARY)

df["detected_terms"] = df["network_text"].apply(detect_terms)
df["term_count"] = df["detected_terms"].apply(len)

print(f"Comments with 0 terms:  {(df['term_count'] == 0).sum():,}")
print(f"Comments with 1 term:   {(df['term_count'] == 1).sum():,}")
print(f"Comments with 2+ terms: {(df['term_count'] >= 2).sum():,}")

# Term frequency overview
all_detected = [t for terms in df["detected_terms"] for t in terms]
term_freq = Counter(all_detected)
print(f"\nTop 20 detected terms:")
for term, count in term_freq.most_common(20):
    print(f"  {term:<22} {count:>5}  ({TERM_THEMES[term]})")


Comments with 0 terms:  4,016
Comments with 1 term:   1,482
Comments with 2+ terms: 743

Top 20 detected terms:
  brainrot                 488  (brainrot)
  social_media             306  (platform)
  genz                     221  (generational)
  slang                    199  (language)
  genalpha                 194  (generational)
  language                 167  (language)
  scrolling                143  (platform)
  shorts                   131  (platform)
  skibidi                  130  (meme_slang)
  tiktok                   125  (platform)
  focus                    112  (cognitive)
  addiction                 73  (cognitive)
  attention_span            73  (cognitive)
  instagram                 68  (platform)
  reels                     58  (platform)
  shortform                 55  (content)
  screen_time               54  (platform)
  dopamine                  50  (cognitive)
  meme                      45  (language)
  algorithm                 38  (platform)


In [7]:
# Keep Comments with at Least 2 Detected Terms

df_network = df[df["term_count"] >= 2].reset_index(drop=True)

print(f"Before filter: {len(df):,} comments")
print(f"After filter (>= 2 terms): {len(df_network):,} comments "
      f"({len(df_network)/len(df)*100:.1f}% of corpus)")
df_network[["network_text", "detected_terms", "term_count"]].head(5)


Before filter: 6,241 comments
After filter (>= 2 terms): 743 comments (11.9% of corpus)


,network_text,detected_terms,term_count
0,as a genz this was my first time feeling compl...,"[genalpha, genz, slang]",3
1,i m writing from argentina and this video reso...,"[anxiety, based, brainrot, meme, overstimulati...",7
2,i was really hoping you d make a video on 6 7 ...,"[culture, slang]",2
3,12 08 please please please please please make ...,"[genz, slang]",2
4,watching your videos makes my brain feel good ...,"[adhd, dopamine]",2


In [8]:
# Generate Term Pairs and Count Edge Weights

edge_counter = Counter()

for terms in df_network["detected_terms"]:
    # combinations gives each unordered pair exactly once per comment
    for pair in combinations(sorted(terms), 2):
        edge_counter[pair] += 1

edges_df = pd.DataFrame(
    [(src, tgt, w) for (src, tgt), w in edge_counter.items()],
    columns=["source", "target", "weight"]
).sort_values("weight", ascending=False).reset_index(drop=True)

print(f"Total unique term pairs (raw edges): {len(edges_df):,}")
print(f"\nWeight distribution:")
print(edges_df["weight"].describe().round(2))
print(f"\nTop 20 edges:")
edges_df.head(20)


Total unique term pairs (raw edges): 472

Weight distribution:
count    472.00
mean       4.05
std        5.70
min        1.00
25%        1.00
50%        2.00
75%        5.00
max       45.00
Name: weight, dtype: float64

Top 20 edges:


,source,target,weight
0,genalpha,genz,45
1,brainrot,genalpha,45
2,genz,slang,45
3,scrolling,social_media,33
4,language,slang,29
5,brainrot,genz,27
6,brainrot,shorts,25
7,genalpha,slang,24
8,brainrot,social_media,24
9,social_media,tiktok,23


In [9]:
# Filter Weak Edges and Remove Isolated Nodes

# Adjust threshold based on the weight distribution shown in Cell 8.
# weight >= 3 keeps more edges (denser graph)
# weight >= 5 keeps fewer edges (cleaner, stronger connections only)
WEIGHT_THRESHOLD = 3

edges_filtered = edges_df[edges_df["weight"] >= WEIGHT_THRESHOLD].copy()
print(f"Edges after weight >= {WEIGHT_THRESHOLD}: {len(edges_filtered):,} "
      f"(removed {len(edges_df) - len(edges_filtered):,})")

# Nodes that appear in at least one retained edge (no isolated nodes)
active_nodes = set(edges_filtered["source"]) | set(edges_filtered["target"])
print(f"\nActive nodes (non-isolated): {len(active_nodes):,}")
print(f"Removed isolated nodes: "
      f"{len(VOCABULARY) - len(active_nodes)} term(s) never co-occur above threshold")
print(f"Isolated terms: {sorted(VOCABULARY - active_nodes)}")


Edges after weight >= 3: 195 (removed 277)

Active nodes (non-isolated): 44
Removed isolated nodes: 10 term(s) never co-occur above threshold
Isolated terms: ['binge', 'boredom', 'brainworm', 'goat', 'overstimulation', 'ratio', 'rewiring', 'sheesh', 'slay', 'zoomer']


In [10]:
# Save edge list and node list

# --- Edge list ---
edges_filtered.to_csv("data_network_analysis.csv", index=False, encoding="utf-8")
print(f"Saved edge list: data_network_analysis.csv  ({len(edges_filtered):,} edges)")

# --- Node list (with theme labels, will be enriched with metrics in Cell 12) ---
nodes_df = pd.DataFrame({
    "node":  sorted(active_nodes),
    "theme": [TERM_THEMES[n] for n in sorted(active_nodes)],
})
nodes_df.to_csv("data_network_analysis_nodes.csv", index=False, encoding="utf-8")
print(f"Saved node list: data_network_analysis_nodes.csv  ({len(nodes_df):,} nodes)")

edges_filtered.head(10)


Saved edge list: data_network_analysis.csv  (195 edges)
Saved node list: data_network_analysis_nodes.csv  (44 nodes)


,source,target,weight
0,genalpha,genz,45
1,brainrot,genalpha,45
2,genz,slang,45
3,scrolling,social_media,33
4,language,slang,29
5,brainrot,genz,27
6,brainrot,shorts,25
7,genalpha,slang,24
8,brainrot,social_media,24
9,social_media,tiktok,23


In [11]:
# Build NetworkX Graph

G = nx.Graph()

# Add nodes with theme attribute
for _, row in nodes_df.iterrows():
    G.add_node(row["node"], theme=row["theme"])

# Add weighted edges
for _, row in edges_filtered.iterrows():
    G.add_edge(row["source"], row["target"], weight=row["weight"])

print(f"Graph summary:")
print(f"  Nodes:   {G.number_of_nodes()}")
print(f"  Edges:   {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")
print(f"  Connected: {nx.is_connected(G)}")
if not nx.is_connected(G):
    components = list(nx.connected_components(G))
    print(f"  Connected components: {len(components)}")
    sizes = sorted([len(c) for c in components], reverse=True)
    print(f"  Component sizes: {sizes}")


Graph summary:
  Nodes:   44
  Edges:   195
  Density: 0.2061
  Connected: True


In [12]:
# Calculate Centrality, Communities and Density

# --- Weighted degree (strength) ---
weighted_degree = dict(G.degree(weight="weight"))

# --- Degree centrality (unweighted, normalised) ---
degree_centrality = nx.degree_centrality(G)

# --- Betweenness centrality ---
# NetworkX treats 'weight' as distance — use inverse weight so
# higher co-occurrence = shorter distance = more central path
for u, v, d in G.edges(data=True):
    G[u][v]["inv_weight"] = 1.0 / d["weight"]

betweenness = nx.betweenness_centrality(G, weight="inv_weight", normalized=True)

# --- Closeness centrality ---
closeness = nx.closeness_centrality(G)

# --- PageRank (weighted) ---
pagerank = nx.pagerank(G, weight="weight", alpha=0.85)

# --- Community detection (greedy modularity, built into NetworkX) ---
communities_raw = list(greedy_modularity_communities(G, weight="weight"))
community_map = {}
for comm_id, members in enumerate(communities_raw):
    for node in members:
        community_map[node] = comm_id

modularity = nx.algorithms.community.quality.modularity(
    G, communities_raw, weight="weight"
)

print(f"Communities detected: {len(communities_raw)}")
print(f"Modularity score:     {modularity:.4f}  (>0.3 indicates meaningful structure)")
print(f"\nCommunity composition:")
for i, comm in enumerate(communities_raw):
    members = sorted(comm)
    print(f"  Community {i}: {members}")

# --- Compile node metrics ---
metrics_df = pd.DataFrame({
    "node":                 list(G.nodes()),
    "theme":                [G.nodes[n].get("theme", "") for n in G.nodes()],
    "community":            [community_map[n] for n in G.nodes()],
    "degree":               [G.degree(n) for n in G.nodes()],
    "weighted_degree":      [weighted_degree[n] for n in G.nodes()],
    "degree_centrality":    [round(degree_centrality[n], 4) for n in G.nodes()],
    "betweenness":          [round(betweenness[n], 4) for n in G.nodes()],
    "closeness":            [round(closeness[n], 4) for n in G.nodes()],
    "pagerank":             [round(pagerank[n], 4) for n in G.nodes()],
}).sort_values("weighted_degree", ascending=False).reset_index(drop=True)

# Overwrite node CSV with full metrics
metrics_df.to_csv("data_network_analysis_nodes.csv", index=False, encoding="utf-8")
print(f"\nNode metrics saved to data_network_analysis_nodes.csv")

print(f"\nTop 15 nodes by weighted degree:")
print(metrics_df[["node","theme","community","degree","weighted_degree",
                   "betweenness","pagerank"]].head(15).to_string(index=False))

print(f"\n--- Graph Statistics ---")
print(f"Nodes:              {G.number_of_nodes()}")
print(f"Edges:              {G.number_of_edges()}")
print(f"Density:            {nx.density(G):.4f}")
print(f"Avg weighted degree:{sum(weighted_degree.values())/len(weighted_degree):.2f}")
print(f"Communities:        {len(communities_raw)}")
print(f"Modularity:         {modularity:.4f}")


Communities detected: 3
Modularity score:     0.3465  (>0.3 indicates meaningful structure)

Community composition:
  Community 0: ['addiction', 'adhd', 'algorithm', 'anxiety', 'attention_span', 'doomscrolling', 'dopamine', 'focus', 'instagram', 'longform', 'mental_health', 'mindless', 'reels', 'screen_time', 'scrolling', 'shortform', 'shorts', 'social_media', 'tiktok', 'zombie']
  Community 1: ['based', 'brainrot', 'culture', 'distraction', 'genalpha', 'genx', 'genz', 'gyatt', 'ipadkids', 'language', 'meme', 'millennial', 'rizz', 'rotting', 'sigma', 'skibidi', 'slang', 'trend', 'viral']
  Community 2: ['boomer', 'nocap', 'sus', 'vibe', 'yeet']

Node metrics saved to data_network_analysis_nodes.csv

Top 15 nodes by weighted degree:
          node        theme  community  degree  weighted_degree  betweenness  pagerank
      brainrot     brainrot          1      31              305       0.5449    0.0897
  social_media     platform          0      24              257       0.4529    0.07